In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Imports

In [2]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.9/705.9 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 22.3 MB/s eta 0:00:00


In [3]:
from contextlib import contextmanager
import time
from collections import namedtuple
import datetime
import os
import cv2
import kagglehub
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from joblib import Parallel, delayed
from tqdm import tqdm
from collections import OrderedDict
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mlflow


# Utils

In [4]:
@contextmanager
def timer(name: str, _align):
    s = time.time()
    yield
    elapsed = time.time() - s
    print(f"{ '[' + name + ']' :{_align}} | {time.strftime('%Y-%m-%d %H:%M:%S')} Done | Using {elapsed: .3f} seconds")

class Dict2ObjParser():
    def __init__(self, nested_dict):
        self.nested_dict = nested_dict

    def parse(self):
        nested_dict = self.nested_dict
        if (obj_type := type(nested_dict)) is not dict:
            raise TypeError(f"Expected 'dict' but found '{obj_type}'")
        return self._transform_to_named_tuples("root", nested_dict)

    def _transform_to_named_tuples(self, tuple_name, possibly_nested_obj):
        if type(possibly_nested_obj) is dict:
            named_tuple_def = namedtuple(tuple_name, possibly_nested_obj.keys())
            transformed_value = named_tuple_def(
                *[
                    self._transform_to_named_tuples(key, value)
                    for key, value in possibly_nested_obj.items()
                ]
            )
        elif type(possibly_nested_obj) is list:
            transformed_value = [
                self._transform_to_named_tuples(f"{tuple_name}_{i}", possibly_nested_obj[i])
                for i in range(len(possibly_nested_obj))
            ]
        else:
            transformed_value = possibly_nested_obj

        return transformed_value

# Dataset

In [24]:
SUPPORTED_INDICATORS = ["MA"]


def cal_indicators(tabular_df, indicator_name, parameters):
    if indicator_name == "MA":
        assert len(parameters) == 1, f'Wrong parameters num, expected 1, got {len(parameters)}'
        slice_win_size = int(parameters[0])
        MA = tabular_df['close'].rolling(slice_win_size, min_periods=1).mean()
        return MA # pd.Series


def single_symbol_latest_image(
    tabular_df, image_size, indicators, show_volume, lookback_days=60
):
    """Generate a single image using the latest lookback_days of data for a symbol

    parameters: [
        tabular_df     -> pandas.DataFrame: tabular data for a single symbol,
        image_size     -> tuple: (H, W), should be (128, 180) for 60-day lookback,
        indicators     -> dict: technical indicators added on the image, e.g. {"MA": [20]},
        show_volume    -> boolean: show Volume bars or not,
        lookback_days  -> int: number of days to look back (default 60)
    ]

    return -> [np.array(image_size), symbol, latest_date] or None if insufficient data
    """

    # Get the latest lookback_days of data
    if len(tabular_df) < lookback_days:
        print(f"Warning: Not enough data for symbol {tabular_df.iloc[0]['Symbol']}. Available: {len(tabular_df)}, Required: {lookback_days}")
        return None

    # Take the latest lookback_days
    latest_data = tabular_df.tail(lookback_days).copy()

    ind_names = []
    if indicators:
        for ind_dict in indicators:
            for ind, params in ind_dict.items():
                assert ind in SUPPORTED_INDICATORS, f"Error: {ind} not supported"
                ind_names.append(ind)
                latest_data[ind] = cal_indicators(
                    latest_data,
                    ind,
                    params if not isinstance(params, list) else params,
                )

    # Check for no transaction days
    if (
        1.0
        * (
            latest_data[["Open", "High", "Low", "Close"]].sum(axis=1)
            / latest_data["Open"]
            == 4
        )
    ).sum() > lookback_days // 5:
        print(f"Warning: Too many no-transaction days for symbol {latest_data.iloc[0]['Symbol']}")
        return None

    # Project price into quantile
    price_slice = latest_data[["Open", "High", "Low", "Close"] + ind_names].reset_index(drop=True)
    volume_slice = latest_data[["Volume"]].reset_index(drop=True)

    price_slice = (price_slice - np.min(price_slice.values)) / (
        np.max(price_slice.values) - np.min(price_slice.values)
    )
    volume_slice = (volume_slice - np.min(volume_slice.values)) / (
        np.max(volume_slice.values) - np.min(volume_slice.values)
    )

    if not show_volume:
        price_slice = price_slice.apply(lambda x: x * (image_size[0] - 1)).astype(int)
    else:
        if image_size[0] == 32:
            price_slice = price_slice.apply(lambda x: x * (25 - 1) + 7).astype(int)
            volume_slice = volume_slice.apply(lambda x: x * (6 - 1)).astype(int)
        else:
            price_slice = price_slice.apply(lambda x: x * (51 - 1) + 13).astype(int)
            volume_slice = volume_slice.apply(lambda x: x * (12 - 1)).astype(int)

    # Create the image
    image = np.zeros(image_size)
    for i in range(len(price_slice)):
        # draw candlestick
        image[price_slice.loc[i]["Open"], i * 3] = 255.0
        image[
            price_slice.loc[i]["Low"] : price_slice.loc[i]["High"] + 1, i * 3 + 1
        ] = 255.0
        image[price_slice.loc[i]["Close"], i * 3 + 2] = 255.0
        # draw indicators
        for ind in ind_names:
            image[price_slice.loc[i][ind], i * 3 : i * 3 + 2] = 255.0
        # draw Volume bars
        if show_volume:
            image[: volume_slice.loc[i]["Volume"], i * 3 + 1] = 255.0

    symbol = latest_data.iloc[-1]["Symbol"]
    latest_date = latest_data.iloc[-1]["Date"]

    return [image, symbol, latest_date]


def single_symbol_image(
    tabular_df, image_size, start_date, sample_rate, indicators, show_volume, mode
):
    """generate Candlelist images

    parameters: [
        tabular_df  -> pandas.DataFrame: tabular data,
        image_size  -> tuple: (H, W), size shouble (32, 15), (64, 60)
        start_date  -> int: truncate extra rows after generating images,
        indicators  -> dict: technical indicators added on the image, e.g. {"MA": [20]},
        show_volume -> boolean: show Volume bars or not
        mode        -> 'train': for train & validation; 'test': for test; 'inference': for inference
    ]

    Note: A single day's data occupies 3 pixel (width). First rows's dates should be prior to the start Date in order to make sure there are enough data to generate image for the start Date.

    return -> list: each item of the list is [np.array(image_size), binary, binary, binary, binary]. The last three binary (0./1.) are the labels of ret5, ret20, ret60

    """

    ind_names = []
    if indicators:
        for i in range(len(indicators)//2):
            ind = indicators[i*2].NAME
            ind_names.append(ind)
            params = str(indicators[i*2+1].PARAM).split(' ')
            tabular_df[ind] = cal_indicators(tabular_df, ind, params)

    dataset = []
    valid_dates = []
    lookback = image_size[1] // 3
    for d in range(lookback - 1, len(tabular_df)):
        # random skip some trading dates
        if np.random.rand(1) > sample_rate:
            continue
        # skip dates before start_date
        if tabular_df.iloc[d]["date"] < start_date:
            continue

        price_slice = tabular_df[d - (lookback - 1) : d + 1][
            ["open", "high", "low", "close"] + ind_names
        ].reset_index(drop=True)
        volume_slice = tabular_df[d - (lookback - 1) : d + 1][["volume"]].reset_index(
            drop=True
        )

        # number of no transactions days > 0.2*look back days
        if (
            1.0
            * (
                price_slice[["open", "high", "low", "close"]].sum(axis=1)
                / price_slice["open"]
                == 4
            )
        ).sum() > lookback // 5:
            continue

        valid_dates.append(
            tabular_df.iloc[d]["date"]
        )  # trading dates surviving the validation

        # project price into quantile
        price_slice = (price_slice - np.min(price_slice.values)) / (
            np.max(price_slice.values) - np.min(price_slice.values)
        )
        volume_slice = (volume_slice - np.min(volume_slice.values)) / (
            np.max(volume_slice.values) - np.min(volume_slice.values)
        )

        if not show_volume:
            price_slice = price_slice.apply(lambda x: x * (image_size[0] - 1)).astype(
                int
            )
        else:
            if image_size[0] == 32:
                price_slice = price_slice.apply(lambda x: x * (25 - 1) + 7).astype(int)
                volume_slice = volume_slice.apply(lambda x: x * (6 - 1)).astype(int)
            else:
                price_slice = price_slice.apply(lambda x: x * (51 - 1) + 13).astype(int)
                volume_slice = volume_slice.apply(lambda x: x * (12 - 1)).astype(int)

        image = np.zeros(image_size)
        for i in range(len(price_slice)):
            # draw candlelist
            image[price_slice.loc[i]["open"], i * 3] = 255.0
            image[
                price_slice.loc[i]["low"] : price_slice.loc[i]["high"] + 1, i * 3 + 1
            ] = 255.0
            image[price_slice.loc[i]["close"], i * 3 + 2] = 255.0
            # draw indicators
            for ind in ind_names:
                image[price_slice.loc[i][ind], i * 3 : i * 3 + 2] = 255.0
            # draw Volume bars
            if show_volume:
                image[: volume_slice.loc[i]["volume"], i * 3 + 1] = 255.0

        label_ret5 = 1 if np.sign(tabular_df.iloc[d]["ret5"]) > 0 else 0
        label_ret20 = 1 if np.sign(tabular_df.iloc[d]["ret20"]) > 0 else 0
        # label_ret60 = 1 if np.sign(tabular_df.iloc[d]["ret60"]) > 0 else 0

        entry = [
            image,
            label_ret5,
            label_ret20,
            # label_ret60,
            # tabular_df.iloc[d]["date"],
            # tabular_df.iloc[d]["code"],
        ]
        dataset.append(entry)

    if mode == "train" or mode == "test":
        return dataset
    else:
        return [tabular_df.iloc[0]["code"], dataset, valid_dates]



class ImageDataSet:
    def __init__(
        self,
        win_size,
        start_date,
        end_date,
        mode,
        label,
        indicators=[],
        show_volume=False,
        parallel_num=-1,
    ):
        ## Check whether inputs are valid
        assert isinstance(start_date, int) and isinstance(
            end_date, int
        ), f"Type Error: start_date & end_date shoule be int"
        assert (
            start_date < end_date
        ), f"start date {start_date} cannnot be later than end date {end_date}"
        assert win_size in [5, 20, 60], f"Wrong look back days: {win_size}"
        assert mode in ["train", "test", "inference"], f"Type Error: {mode}"
        assert label in ["RET5", "RET20"], f"Wrong Label: {label}"
        assert indicators is None or len(indicators)%2 == 0, 'Config Error, length of indicators should be even'
        if indicators:
            for i in range(len(indicators)//2):
                assert indicators[2*i].NAME in SUPPORTED_INDICATORS, f"Error: Calculation of {indicators[2*i].NAME} is not defined"

        ## Attributes of ImageDataSet
        if win_size == 5:
            self.image_size = (32, 15)
            self.extra_dates = datetime.timedelta(days=40)
        elif win_size == 20:
            self.image_size = (64, 60)
            self.extra_dates = datetime.timedelta(days=40)
        else:
            self.image_size = (128, 180)
            self.extra_dates = datetime.timedelta(days=60)

        self.start_date = start_date
        self.end_date = end_date
        self.mode = mode
        self.label = label
        self.indicators = indicators
        self.show_volume = show_volume
        self.parallel_num = parallel_num

        ## Load data from zipfile
        self.load_data()

        # Log info
        if indicators:
            ind_info = [(self.indicators[2*i].NAME, str(self.indicators[2*i+1].PARAM).split(' ')) for i in range(len(self.indicators)//2)]
        else:
            ind_info = []
        print(
            f"DataSet Initialized\n \t - Mode:         {self.mode.upper()}\n \t - Image Size:   {self.image_size}\n \t - Time Period:  {self.start_date} - {self.end_date}\n \t - Indicators:   {indicators}\n \t - Volume Shown: {self.show_volume}"
        )

    @timer("Load Data", "8")
    def load_data(self):
        path = "/content/drive/MyDrive/CNN-for-Trading/data/"
        print("Path to dataset files:", path)

        # Find CSV file
        csv_file = None
        for file in os.listdir(path):
            if file.endswith(".csv"):
                csv_file = os.path.join(path, file)
                break
        if not csv_file:
            print("No CSV file found in the downloaded dataset.")
            return None

        # Load CSV
        tabularDf = pd.read_csv(csv_file)
        print(f"Loaded dataset with shape: {tabularDf.shape}")

        # Parse 'Date' column to datetime
        tabularDf["date"] = pd.to_datetime(tabularDf["date"], format="%d-%m-%Y")

        # Padding for extra dates
        padding_start_date = pd.to_datetime(str(self.start_date)) - self.extra_dates
        padding_end_date = pd.to_datetime(str(self.end_date)) + self.extra_dates

        self.df = tabularDf.loc[
            (tabularDf["date"] > padding_start_date)
            & (tabularDf["date"] < padding_end_date)
        ].copy(deep=False)
        tabularDf = []  # clear memory

        self.df["ret5"] = np.zeros(self.df.shape[0])
        self.df["ret20"] = np.zeros(self.df.shape[0])
        self.df["ret5"] = (self.df["close"].pct_change(5) * 100).shift(-5)
        self.df["ret20"] = (self.df["close"].pct_change(20) * 100).shift(-20)
        self.start_date = pd.to_datetime(str(self.start_date))
        self.end_date = pd.to_datetime(str(self.end_date))
        self.df = self.df.loc[self.df["date"] <= self.end_date]

    def generate_images(self, sample_rate):
        dataset_all = Parallel(n_jobs=self.parallel_num)(
            delayed(single_symbol_image)(
                g[1],
                image_size=self.image_size,
                start_date=self.start_date,
                sample_rate=sample_rate,
                indicators=self.indicators,
                show_volume=self.show_volume,
                mode=self.mode,
            )
            for g in tqdm(
                self.df.groupby("code"),
                desc=f"Generating Images (sample rate: {sample_rate})",
            )
        )

        if self.mode == "train" or self.mode == "test":
            image_set = []
            for symbol_data in dataset_all:
                image_set = image_set + symbol_data
            dataset_all = []  # clear memory

            if self.mode == "train":  # resample to handle imbalance
                image_set = pd.DataFrame(
                    image_set, columns=["img", "ret5", "ret20"] #, "date", "code"]
                )
                image_set["index"] = image_set.index
                smote = SMOTE()
                if self.label == "RET5":
                    num0_before = image_set.loc[image_set["ret5"] == 0].shape[0]
                    num1_before = image_set.loc[image_set["ret5"] == 1].shape[0]
                    resample_index, _ = smote.fit_resample(
                        image_set[["index", "ret20"]], image_set["ret5"]
                    )
                    image_set = image_set[["img", "ret5", "ret20"]].loc[ #"date", "code"
                        resample_index["index"]
                    ]
                    num0 = image_set.loc[image_set["ret5"] == 0].shape[0]
                    num1 = image_set.loc[image_set["ret5"] == 1].shape[0]
                    image_set = image_set.values.tolist()

                else:
                    num0_before = image_set.loc[image_set["ret20"] == 0].shape[0]
                    num1_before = image_set.loc[image_set["ret20"] == 1].shape[0]
                    resample_index, _ = smote.fit_resample(
                        image_set[["index", "ret5"]], image_set["ret20"]
                    )
                    image_set = image_set[["img", "ret5", "ret20"]].loc[ #"date", "code"
                        resample_index["index"]
                    ]
                    num0 = image_set.loc[image_set["ret20"] == 0].shape[0]
                    num1 = image_set.loc[image_set["ret20"] == 1].shape[0]
                    image_set = image_set.values.tolist()

                print(
                    f"LABEL: {self.label}\n\tBefore Resample: 0: {num0_before}/{num0_before+num1_before}, 1: {num1_before}/{num0_before+num1_before}\n\tResampled ImageSet: 0: {num0}/{num0+num1}, 1: {num1}/{num0+num1}"
                )

            return image_set

        else:
            return dataset_all


# Models

In [6]:
# each day is displayed in 3 pixels
# use cnn to predict
class CNN5d(nn.Module):
    # Input: [N, (1), 32, 15]; Output: [N, 2]
    # Two Convolution Blocks

    def init_weights(self, m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
            torch.nn.init.xavier_uniform(m.weight)
            m.bias.data.fill_(0.01)

    def __init__(self):
        super(CNN5d, self).__init__()
        self.conv1 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(1, 64, (5, 3), padding=(2, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 64, 32, 15]
            ('BN', nn.BatchNorm2d(64, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 64, 16, 15]
        ]))
        self.conv1 = self.conv1.apply(self.init_weights)

        self.conv2 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(64, 128, (5, 3), padding=(2, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 128, 16, 15]
            ('BN', nn.BatchNorm2d(128, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 128, 8, 15]
        ]))
        self.conv2 = self.conv2.apply(self.init_weights)

        self.DropOut = nn.Dropout(p=0.5)
        self.FC = nn.Linear(15360, 2)
        self.init_weights(self.FC)
        self.Softmax = nn.Softmax(dim=1)

    def forward(self, x): # input: [N, 32, 15]
        x = x.unsqueeze(1).to(torch.float32)   # output size: [N, 1, 32, 15]
        x = self.conv1(x) # output size: [N, 64, 16, 15]
        x = self.conv2(x) # output size: [N, 128, 8, 15]
        x = self.DropOut(x.view(x.shape[0], -1))
        x = self.FC(x) # output size: [N, 2]
        x = self.Softmax(x)

        return x



class CNN20d(nn.Module):
    # Input: [N, (1), 64, 60]; Output: [N, 2]
    # Three Convolution Blocks

    def init_weights(self, m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
            torch.nn.init.xavier_uniform(m.weight)
            m.bias.data.fill_(0.01)

    def __init__(self):
        super(CNN20d, self).__init__()
        self.conv1 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(1, 64, (5, 3), padding=(3, 1), stride=(3, 1), dilation=(2, 1))), # output size: [N, 64, 21, 60]
            ('BN', nn.BatchNorm2d(64, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 64, 10, 60]
        ]))
        self.conv1 = self.conv1.apply(self.init_weights)

        self.conv2 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(64, 128, (5, 3), padding=(3, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 128, 12, 60]
            ('BN', nn.BatchNorm2d(128, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 128, 6, 60]
        ]))
        self.conv2 = self.conv2.apply(self.init_weights)

        self.conv3 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(128, 256, (5, 3), padding=(2, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 256, 6, 60]
            ('BN', nn.BatchNorm2d(256, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 256, 3, 60]
        ]))
        self.conv3 = self.conv3.apply(self.init_weights)

        self.DropOut = nn.Dropout(p=0.5)
        self.FC = nn.Linear(46080, 2)
        self.init_weights(self.FC)
        self.Softmax = nn.Softmax(dim=1)

    def forward(self, x): # input: [N, 64, 60]
        x = x.unsqueeze(1).to(torch.float32)   # output size: [N, 1, 64, 60]
        x = self.conv1(x) # output size: [N, 64, 10, 60]
        x = self.conv2(x) # output size: [N, 128, 6, 60]
        x = self.conv3(x) # output size: [N, 256, 3, 60]
        x = self.DropOut(x.view(x.shape[0], -1))
        x = self.FC(x) # output size: [N, 2]
        x = self.Softmax(x)

        return x


class CNN60d(nn.Module):
    # Input: [N, (1), 128, 180]; Output: [N, 2]
    # Four Convolution Blocks

    def init_weights(self, m):
        if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
            torch.nn.init.xavier_uniform(m.weight)
            m.bias.data.fill_(0.01)

    def __init__(self):
        super(CNN60d, self).__init__()
        self.conv1 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(1, 64, (5, 3), padding=(4, 1), stride=(4, 1), dilation=(3, 1))), # output size: [N, 64, 32, 180]
            ('BN', nn.BatchNorm2d(64, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 64, 16, 180]
        ]))
        self.conv1 = self.conv1.apply(self.init_weights)

        self.conv2 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(64, 128, (5, 3), padding=(4, 1), stride=(1, 1), dilation=(2, 1))), # output size: [N, 128, 16, 180]
            ('BN', nn.BatchNorm2d(128, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 128, 8, 180]
        ]))
        self.conv2 = self.conv2.apply(self.init_weights)

        self.conv3 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(128, 256, (5, 3), padding=(3, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 256, 10, 180]
            ('BN', nn.BatchNorm2d(256, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 256, 5, 180]
        ]))
        self.conv3 = self.conv3.apply(self.init_weights)

        self.conv4 = nn.Sequential(OrderedDict([
            ('Conv', nn.Conv2d(256, 512, (5, 3), padding=(2, 1), stride=(1, 1), dilation=(1, 1))), # output size: [N, 512, 5, 180]
            ('BN', nn.BatchNorm2d(512, affine=True)),
            ('ReLU', nn.ReLU()),
            ('Max-Pool', nn.MaxPool2d((2,1))) # output size: [N, 512, 2, 180]
        ]))
        self.conv4 = self.conv4.apply(self.init_weights)

        self.DropOut = nn.Dropout(p=0.5)
        self.FC = nn.Linear(184320, 2)
        self.init_weights(self.FC)
        self.Softmax = nn.Softmax(dim=1)

    def forward(self, x): # input: [N, 128, 180]
        x = x.unsqueeze(1).to(torch.float32)   # output size: [N, 1, 128, 180]
        x = self.conv1(x) # output size: [N, 64, 16, 180]
        x = self.conv2(x) # output size: [N, 128, 8, 180]
        x = self.conv3(x) # output size: [N, 256, 5, 180]
        x = self.conv4(x) # output size: [N, 512, 2, 180]
        x = self.DropOut(x.view(x.shape[0], -1))
        x = self.FC(x) # output size: [N, 2]
        x = self.Softmax(x)

        return x

# Training

In [27]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_n_epochs(n_epochs, model, label_type, train_loader, valid_loader, criterion, optimizer, savefile, early_stop_epoch, setting=None):
    valid_loss_min = np.inf
    train_loss_set, valid_loss_set = [], []
    train_acc_set, valid_acc_set = [], []
    invariant_epochs = 0

    # Start MLflow run
    with mlflow.start_run():

        # Log hyperparameters if provided
        if setting:
            mlflow.log_params({
                "n_epochs": n_epochs,
                "learning_rate": setting.TRAIN.LEARNING_RATE,
                "weight_decay": setting.TRAIN.WEIGHT_DECAY,
                "batch_size": setting.TRAIN.BATCH_SIZE,
                "label_type": label_type,
                "early_stop_epoch": early_stop_epoch,
                "model": setting.MODEL
            })

        for epoch_i in range(1, n_epochs+1):
            # Training loop
            train_loss, train_acc = 0.0, 0.0
            valid_loss, valid_acc = 0.0, 0.0

            model.train()
            for data, ret5, ret20 in train_loader:
                target = ret5 if label_type == "RET5" else ret20
                target = (1-target).unsqueeze(1) @ torch.LongTensor([1., 0.]).unsqueeze(1).T \
                         + target.unsqueeze(1) @ torch.LongTensor([0, 1]).unsqueeze(1).T
                target = target.to(torch.float32)

                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * data.size(0)
                train_acc += (output.argmax(1) == target.argmax(1)).sum()

            # Validation loop
            model.eval()
            with torch.no_grad():
                for data, ret5, ret20 in valid_loader:
                    target = ret5 if label_type == "RET5" else ret20
                    target = (1-target).unsqueeze(1) @ torch.LongTensor([1., 0.]).unsqueeze(1).T \
                             + target.unsqueeze(1) @ torch.LongTensor([0, 1]).unsqueeze(1).T
                    target = target.to(torch.float32)

                    data, target = data.to(device), target.to(device)
                    output = model(data)
                    loss = criterion(output, target)
                    valid_loss += loss.item() * data.size(0)
                    valid_acc += (output.argmax(1) == target.argmax(1)).sum()

            # Average losses and acc
            train_loss /= len(train_loader.sampler)
            valid_loss /= len(valid_loader.sampler)
            train_acc = train_acc / len(train_loader.sampler)
            valid_acc = valid_acc / len(valid_loader.sampler)

            train_loss_set.append(train_loss)
            valid_loss_set.append(valid_loss)
            train_acc_set.append(train_acc.cpu().numpy())
            valid_acc_set.append(valid_acc.cpu().numpy())

            print(f"Epoch: {epoch_i} Training Loss: {train_loss:.6f} Validation Loss: {valid_loss:.6f} Training Acc: {train_acc:.5f} Validation Acc: {valid_acc:.5f}")

            # Log metrics to MLflow
            mlflow.log_metrics({
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "train_acc": float(train_acc),
                "valid_acc": float(valid_acc)
            }, step=epoch_i)

            # Save model if val improved
            if valid_loss <= valid_loss_min:
                print(f"Validation loss decreased ({valid_loss_min:.6f} --> {valid_loss:.6f}). Saving model ...")
                valid_loss_min = valid_loss
                invariant_epochs = 0
                torch.save({
                    "epoch": epoch_i,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict()
                }, savefile)

                # Log model checkpoint to MLflow
                mlflow.pytorch.log_model(model, "model")
            else:
                invariant_epochs += 1

            if invariant_epochs >= early_stop_epoch:
                print(f"Early Stop at Epoch [{epoch_i}]")
                break

    return train_loss_set, valid_loss_set, train_acc_set, valid_acc_set


def plot_loss_and_acc(loss_and_acc_dict, mlflow_log=False):
    _, axes = plt.subplots(1, 2, figsize=(20, 6))
    tmp = list(loss_and_acc_dict.values())
    maxEpoch = len(tmp[0][0])

    maxLoss = max([max(x[0]) for x in loss_and_acc_dict.values()]) + 0.1
    minLoss = max(0, min([min(x[0]) for x in loss_and_acc_dict.values()]) - 0.1)

    for name, lossAndAcc in loss_and_acc_dict.items():
        axes[0].plot(range(1, 1 + maxEpoch), lossAndAcc[0], '-s', label=name)

    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_xticks(range(0, maxEpoch + 1, maxEpoch//10))
    axes[0].axis([0, maxEpoch, minLoss, maxLoss])
    axes[0].legend()
    axes[0].set_title("Error")

    maxAcc = min(1, max([max(x[1]) for x in loss_and_acc_dict.values()]) + 0.1)
    minAcc = max(0, min([min(x[1]) for x in loss_and_acc_dict.values()]) - 0.1)

    for name, lossAndAcc in loss_and_acc_dict.items():
        axes[1].plot(range(1, 1 + maxEpoch), lossAndAcc[1], '-s', label=name)

    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_xticks(range(0, maxEpoch + 1, maxEpoch//10))
    axes[1].axis([0, maxEpoch, minAcc, maxAcc])
    axes[1].legend()
    axes[1].set_title("Accuracy")

    if mlflow_log:
        import io
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        mlflow.log_image(buf, "loss_accuracy.png")
        buf.close()

# Main Training Function

In [28]:
import os
import sys
import argparse
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

setting_path = "/content/drive/MyDrive/CNN-for-Trading/configs/I60R5/I60R5_18-20.yml"
output_dir = "/content/drive/MyDrive/CNN-for-Trading/generated_images"
os.makedirs(output_dir, exist_ok=True)

with open(setting_path, 'r') as f:
    setting = Dict2ObjParser(yaml.safe_load(f)).parse()

os.makedirs("/content/drive/MyDrive/CNN-for-Trading/models", exist_ok=True)
os.makedirs("/content/drive/MyDrive/CNN-for-Trading/logs", exist_ok=True)

model_dir = os.path.join("/content/drive/MyDrive/CNN-for-Trading/models", setting.TRAIN.MODEL_SAVE_FILE.split('/')[1])
log_dir   = os.path.join("/content/drive/MyDrive/CNN-for-Trading/logs",   setting.TRAIN.LOG_SAVE_FILE.split('/')[1])

os.makedirs(model_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

save_model_path = os.path.join("/content/drive/MyDrive/CNN-for-Trading/", setting.TRAIN.MODEL_SAVE_FILE)
save_log_path   = os.path.join("/content/drive/MyDrive/CNN-for-Trading/", setting.TRAIN.LOG_SAVE_FILE)

if os.path.exists(save_model_path):
    print(f"Pretrained Model: {setting_path} already exists.")
    sys.exit(0)

dataset = ImageDataSet(
    win_size=setting.DATASET.LOOKBACK_WIN,
    start_date=setting.DATASET.START_DATE,
    end_date=setting.DATASET.END_DATE,
    mode='train',
    label=setting.TRAIN.LABEL,
    indicators=setting.DATASET.INDICATORS,
    show_volume=setting.DATASET.SHOW_VOLUME,
    parallel_num=setting.DATASET.PARALLEL_NUM
)

image_set = dataset.generate_images(setting.DATASET.SAMPLE_RATE)
# for i, (img, ret5, ret20, date, code) in enumerate(image_set):
#         if isinstance(date, pd.Timestamp):
#             date_str = date.strftime("%Y%m%d")
#         else:
#             date_str = str(date)
#         filename = (
#             f"{output_dir}/{code}_{date_str}_img_{i}_ret5_{ret5}_ret20_{ret20}.png"
#         )
#         success = cv2.imwrite(filename, img)
#         if not success:
#             print(f"Failed to save {filename}, shape={img.shape}, dtype={img.dtype}")

train_loader_size = int(len(image_set) * (1 - setting.TRAIN.VALID_RATIO))
valid_loader_size = len(image_set) - train_loader_size

train_loader, valid_loader = torch.utils.data.random_split(
    image_set, [train_loader_size, valid_loader_size]
)
train_loader = torch.utils.data.DataLoader(
    dataset=train_loader, batch_size=setting.TRAIN.BATCH_SIZE, shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    dataset=valid_loader, batch_size=setting.TRAIN.BATCH_SIZE, shuffle=True
)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert setting.MODEL in ['CNN5d', 'CNN20d', 'CNN60d'], f"Wrong Model Template: {setting.MODEL}"

if __name__ == '__main__':

    if setting.MODEL == 'CNN5d':
        model = CNN5d()
    elif setting.MODEL == 'CNN20d':
        model = CNN20d()
    else:
      model = CNN60d()
    model.to(device)

    criterion = nn.BCELoss().to(device)
    optimizer = optim.Adam(
        model.parameters(),
        lr=setting.TRAIN.LEARNING_RATE,
        weight_decay=setting.TRAIN.WEIGHT_DECAY
    )

    train_loss_set, valid_loss_set, train_acc_set, valid_acc_set = train_n_epochs(
        setting.TRAIN.NEPOCH,
        model,
        setting.TRAIN.LABEL,
        train_loader,
        valid_loader,
        criterion,
        optimizer,
        save_model_path,
        setting.TRAIN.EARLY_STOP_EPOCH
    )

    log = pd.DataFrame(
        [train_loss_set, train_acc_set, valid_loss_set, valid_acc_set],
        index=['train_loss', 'train_acc', 'valid_loss', 'valid_acc']
    )
    log.to_csv(save_log_path)


Path to dataset files: /content/drive/MyDrive/CNN-for-Trading/data/
Loaded dataset with shape: (728197, 7)
[Load Data] | 2025-09-16 07:36:50 Done | Using  2.134 seconds
DataSet Initialized
 	 - Mode:         TRAIN
 	 - Image Size:   (128, 180)
 	 - Time Period:  2018-12-31 00:00:00 - 2020-12-31 00:00:00
 	 - Indicators:   [INDICATORS_0(NAME='MA'), INDICATORS_1(PARAM=5)]
 	 - Volume Shown: True



Generating Images (sample rate: 0.2): 100%|██████████| 198/198 [03:42<00:00,  1.12s/it]


LABEL: RET5
	Before Resample: 0: 8772/18433, 1: 9661/18433
	Resampled ImageSet: 0: 9441/19322, 1: 9881/19322


/tmp/ipython-input-3547496812.py:105: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(m.weight)
2025/09/16 07:41:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/16 07:41:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch: 1 Training Loss: 1.120628 Validation Loss: 0.795867 Training Acc: 0.50713 Validation Acc: 0.51751
Validation loss decreased (inf --> 0.795867). Saving model ...


...

Epoch: 2 Training Loss: 0.995782 Validation Loss: 0.784255 Training Acc: 0.53811 Validation Acc: 0.53183
Validation loss decreased (0.795867 --> 0.784255). Saving model ...


2025/09/16 07:43:19 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/09/16 07:43:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/09/16 07:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch: 3 Training Loss: 0.927214 Validation Loss: 0.759795 Training Acc: 0.57494 Validation Acc: 0.53786
Validation loss decreased (0.784255 --> 0.759795). Saving model ...


2025/09/16 07:44:34 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/09/16 07:44:39 WARNING mlflow.utils.requirements_utils: Found torch version (2.8.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.8.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/09/16 07:44:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Epoch: 4 Training Loss: 0.856851 Validation Loss: 0.772927 Training Acc: 0.59601 Validation Acc: 0.54563
Epoch: 5 Training Loss: 0.784578 Validation Loss: 0.771022 Training Acc: 0.63201 Validation Acc: 0.54683
Epoch: 6 Training Loss: 0.721114 Validation Loss: 0.777888 Training Acc: 0.65774 Validation Acc: 0.54545
Epoch: 7 Training Loss: 0.685786 Validation Loss: 0.779702 Training Acc: 0.67046 Validation Acc: 0.54908
Early Stop at Epoch [7]


In [ ]:
import torch

checkpoint = torch.load(
    '/content/drive/MyDrive/CNN-for-Trading/models/I60R5_OHLC/I60R5_OHLC_18-20.tar',
    map_location=torch.device('cpu')
)

# Inspect keys if it's a dict
print(checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict'])
